# Amazon Mobile Phone Reviews — Preprocessing & Model Extraction

**Pipeline overview:**
1. Load raw data & exploratory analysis
2. Clean and filter reviews
3. Extract unique product names
4. LangExtract — one-time brand/model extraction with checkpoint system
5. Save final cleaned dataset

## 1. Setup & Imports

In [ ]:
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import langextract as lx

sns.set_theme(style='whitegrid')

RAW_DATA_PATH      = 'data/raw/amazon_reviews_unlocked_mobile_phones.csv'
CHECKPOINT_PATH    = 'data/processed/model_cache.csv'
OUTPUT_PATH        = 'data/processed/reviews_cleaned.csv'
BATCH_SIZE         = 8    # parallel workers per batch
MAX_WORKERS        = 8    # ThreadPoolExecutor concurrency

## 2. Load Data & Exploratory Analysis

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

In [ ]:
# Missing values
missing = df.isnull().sum().rename('missing').to_frame()
missing['pct'] = (missing['missing'] / len(df) * 100).round(2)
missing

In [ ]:
# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Review length distribution
df['review_word_count'] = df['Reviews'].dropna().apply(lambda x: len(str(x).split()))
df['review_word_count'].clip(upper=300).plot(kind='hist', bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Review Word Count Distribution (clipped at 300)')
axes[1].set_xlabel('Word Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 brands by review count
top_brands = df['Brand Name'].value_counts().head(20)

plt.figure(figsize=(12, 5))
top_brands.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 20 Brands by Review Count')
plt.xlabel('Brand')
plt.ylabel('Reviews')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 products by review count
top_products = df['Product Name'].value_counts().head(20)

plt.figure(figsize=(14, 5))
top_products.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 20 Products by Review Count')
plt.xlabel('Product')
plt.ylabel('Reviews')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Clean & Filter Reviews

In [ ]:
def is_too_short(text: str, min_words: int = 8) -> bool:
    if not isinstance(text, str):
        return True
    return len(text.split()) < min_words

def is_question(text: str) -> bool:
    if not isinstance(text, str):
        return False
    text = text.strip()
    if text.endswith('?'):
        return True
    question_words = ('how', 'why', 'can', 'does', 'is', 'what', 'will', 'where', 'when', 'who')
    if text.lower().startswith(question_words):
        return True
    return False

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def should_keep_review(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    if is_too_short(text):
        return False
    if is_question(text):
        return False
    return True

In [ ]:
initial_count = len(df)

# Drop rows with missing reviews
df = df.dropna(subset=['Reviews'])

# Drop rows with missing Brand Name — no point extracting models for unidentifiable products
df = df.dropna(subset=['Brand Name'])

# Clean text
df['Reviews'] = df['Reviews'].apply(clean_text)

# Apply heuristic filters
mask = df['Reviews'].apply(should_keep_review)
df = df[mask].copy()

final_count = len(df)
removed_count = initial_count - final_count

print(f'Initial rows  : {initial_count:,}')
print(f'After cleaning: {final_count:,}')
print(f'Removed       : {removed_count:,} ({100 * removed_count / initial_count:.1f}%)')

## 4. Unique Product Names

In [ ]:
unique_products = df['Product Name'].dropna().unique().tolist()
print(f'Unique product names: {len(unique_products):,}')
unique_products[:10]

## 5. LangExtract — Model & Brand Extraction (with Checkpoint)

In [ ]:
import textwrap

EXTRACTION_PROMPT = textwrap.dedent("""\
    Extract the phone brand and phone model from the product name.
    Use the exact substring from the product name as extraction_text.
    Return null brand and null model if you cannot confidently identify them.
    Do not paraphrase or invent values not present in the text.""")

EXTRACTION_EXAMPLES = [
    lx.data.ExampleData(
        text='Samsung Galaxy S7 Edge 32GB Unlocked GSM 4G LTE Phone',
        extractions=[
            lx.data.Extraction(extraction_class='brand', extraction_text='Samsung',
                               attributes={'confidence': 'high'}),
            lx.data.Extraction(extraction_class='model', extraction_text='Galaxy S7 Edge',
                               attributes={'confidence': 'high'}),
        ]
    ),
    lx.data.ExampleData(
        text='Apple iPhone 6 Plus 16GB Factory Unlocked GSM Smartphone',
        extractions=[
            lx.data.Extraction(extraction_class='brand', extraction_text='Apple',
                               attributes={'confidence': 'high'}),
            lx.data.Extraction(extraction_class='model', extraction_text='iPhone 6 Plus',
                               attributes={'confidence': 'high'}),
        ]
    ),
    lx.data.ExampleData(
        text='Unlocked Android 4G LTE Dual SIM Smartphone 5.5 inch HD',
        extractions=[]  # No identifiable brand or model
    ),
]

In [ ]:
def load_checkpoint() -> pd.DataFrame:
    """Load existing checkpoint if it exists, otherwise return empty DataFrame."""
    if os.path.exists(CHECKPOINT_PATH):
        print(f'Checkpoint found. Loading...')
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=['product_name', 'brand_llm', 'model', 'processed'])


def save_batch_to_checkpoint(batch_results: list[dict]):
    """Append a batch of results to the checkpoint CSV."""
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    batch_df = pd.DataFrame(batch_results)
    header = not os.path.exists(CHECKPOINT_PATH)
    batch_df.to_csv(CHECKPOINT_PATH, mode='a', index=False, header=header)


def parse_extractions(result) -> dict:
    """Parse LangExtract result into {brand_llm, model} dict."""
    brand, model = None, None
    for extraction in result.extractions:
        if extraction.extraction_class == 'brand' and brand is None:
            brand = extraction.extraction_text
        elif extraction.extraction_class == 'model' and model is None:
            model = extraction.extraction_text
    return {'brand_llm': brand, 'model': model}


def extract_phone_info(product_name: str) -> dict:
    """Call LangExtract to extract brand and model from a product name string."""
    result = lx.extract(
        text_or_documents=product_name,
        prompt_description=EXTRACTION_PROMPT,
        examples=EXTRACTION_EXAMPLES,
        model_id='gemini-2.5-flash',
    )
    return parse_extractions(result)

In [ ]:
checkpoint_df = load_checkpoint()

already_processed = set(checkpoint_df['product_name'].tolist()) if not checkpoint_df.empty else set()
to_process = [p for p in unique_products if p not in already_processed]

print(f'Total unique products : {len(unique_products):,}')
print(f'Already processed     : {len(already_processed):,}')
print(f'Remaining             : {len(to_process):,}')

In [ ]:
# Process in batches — each batch runs in parallel, checkpoint written after every batch
batches = [to_process[i:i + BATCH_SIZE] for i in range(0, len(to_process), BATCH_SIZE)]

for batch in tqdm(batches, desc='Batches'):
    batch_results = [None] * len(batch)  # preserve order

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(extract_phone_info, product_name): idx
            for idx, product_name in enumerate(batch)
        }
        for future in as_completed(futures):
            idx = futures[future]
            product_name = batch[idx]
            try:
                info = future.result()
                batch_results[idx] = {
                    'product_name': product_name,
                    'brand_llm': info['brand_llm'],
                    'model': info['model'],
                    'processed': True
                }
            except Exception as e:
                print(f'  [ERROR] "{product_name[:60]}" — {e}')
                batch_results[idx] = {
                    'product_name': product_name,
                    'brand_llm': None,
                    'model': None,
                    'processed': False
                }

    save_batch_to_checkpoint(batch_results)

print('Extraction complete.')

In [ ]:
# Reload full checkpoint and merge into main dataframe
checkpoint_df = pd.read_csv(CHECKPOINT_PATH)

print(f'Checkpoint rows : {len(checkpoint_df):,}')
print(f'Processed       : {checkpoint_df["processed"].sum():,}')
print(f'Failed          : {(~checkpoint_df["processed"]).sum():,}')
print(f'Model null      : {checkpoint_df["model"].isna().sum():,}')

checkpoint_df.head()

In [ ]:
# Merge — keep only the Model column from LangExtract; original Brand Name stays untouched
df = df.merge(
    checkpoint_df[['product_name', 'model']],
    left_on='Product Name',
    right_on='product_name',
    how='left'
).drop(columns=['product_name'])

print(f'Final dataset shape: {df.shape}')
df[['Product Name', 'Brand Name', 'model']].drop_duplicates().head(10)

## 6. Save Final Dataset

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.drop(columns=['review_word_count'], errors='ignore').to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(df):,} rows to {OUTPUT_PATH}')
df.head()